# Methane–ethane phase equilibrium


**Evidence: numerical verification.** A bubble point fixes liquid composition
and solves pressure; a flash fixes temperature, pressure and total feed and
solves phase amounts and compositions.

Read `gross-2001-methane-ethane/3`, based on Gross and Sadowski (2001), Table 2
([paper](https://doi.org/10.1021/ie0003887)). Retain the packet's mixture inputs.
The bubble calculation uses 200 K and liquid mole fractions 0.3/0.7.
The flash uses 250 K, 3.75 MPa and 0.3 mol methane + 0.7 mol ethane.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Mixture,
)
from epcsaft import equilibrium as eq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


def equilibrium_values(solved):
    """Keep the returned physical state and the equation residuals together."""
    assert solved.success, solved.message
    assert solved.requested_tolerance_met
    assert solved.classified_residuals
    residual = max(map(abs, solved.classified_residuals), default=0)
    assert residual <= 1e-10
    return dict(
        temperature_K=solved.temperature,
        pressure_Pa=solved.pressure,
        amounts_mol=solved.amounts,
        mole_fractions=solved.mole_fractions,
        molar_densities_mol_m3=solved.molar_densities,
        residual=residual,
        linear_solver=solved.linear_solver,
    )


## Declare both equilibrium problems

The incipient vapor in the bubble declaration has zero amount. In the flash,
both phase amounts are unknown. Arrays are phase-major: two component values
for vapor followed by two for liquid, in methane/ethane order.

Check bubble pressure and flash vapor amount against the retained numerical
baselines at 10⁻¹⁰ relative, and each component balance within 10⁻¹⁰ mol.
Classified equation residuals must also be at most 10⁻¹⁰.


In [2]:
model = Mixture(
    Parameters.from_json(
        PACKETS / "gross-2001-methane-ethane/3/parameters/parameters.json",
        components=["methane", "ethane"],
    )
)
bubble = eq.solve_equilibrium(
    model,
    eq.bubble_point(
        liquid=eq.Phase("L", kind="liquid"),
        T=200,
        feed=eq.MoleFractions({"methane": 0.3, "ethane": 0.7}),
    ),
)
flash = eq.solve_equilibrium(
    model,
    eq.Problem(
        phases=[eq.Phase("V", kind="vapor"), eq.Phase("L", kind="liquid")],
        T=250,
        P=3750000.0,
        feed=eq.Amounts({"methane": 0.3, "ethane": 0.7}),
    ),
)
closure = [
    sum((flash.amounts[j] * flash.mole_fractions[2 * j + i] for j in range(2))) - v
    for i, v in enumerate((0.3, 0.7))
]
assert max(map(abs, closure)) < 1e-10
assert abs(bubble.pressure / 1695742.8953910777 - 1) < 1e-10
assert abs(flash.amounts[0] / 0.16465931077561355 - 1) < 1e-10
values = dict(
    bubble=equilibrium_values(bubble),
    flash=equilibrium_values(flash),
    component_balance_mol=closure,
)
display(
    pd.DataFrame(
        {
            "Phase": ["Vapor", "Liquid"],
            "Amount / mol": flash.amounts,
            "x methane / 1": flash.mole_fractions[::2],
            "x ethane / 1": flash.mole_fractions[1::2],
        }
    )
)


,Phase,Amount / mol,x methane / 1,x ethane / 1
0,Vapor,0.164659,0.591550,0.408450
1,Liquid,0.835341,0.242531,0.757469


In [3]:
display(pd.DataFrame(
    [values["bubble"], values["flash"]], index=["Bubble", "Flash"]
).rename(columns={
            "temperature_K": "Temperature / K",
            "pressure_Pa": "Pressure / Pa",
            "amounts_mol": "Phase amounts / mol",
            "mole_fractions": "Mole fractions / 1",
            "molar_densities_mol_m3": "Phase densities / mol m^-3",
            "residual": "Maximum classified residual / 1",
            "linear_solver": "Linear solver",
        }))
display(pd.DataFrame({
    "Component": ["methane", "ethane"],
    "Flash component balance error / mol": closure,
}))


,Temperature / K,Pressure / Pa,Phase amounts / mol,Mole fractions / 1,Phase densities / mol m^-3,Maximum classified residual / 1,Linear solver
Bubble,200.0,1.695743e+06,"[1.0, 0.0]","[0.3, 0.7, 0.8673216367151354, 0.13267836328486465]","[18178.421861210227, 1205.1400298939852]",1.776357e-15,
Flash,250.0,3.750000e+06,"[0.16465931077561355, 0.8353406892243862]","[0.591550247299066, 0.4084497527009339, 0.24253067832563274, 0.7574693216743672]","[2561.9512915879263, 14724.75225013111]",2.038041e-15,


,Component,Flash component balance error / mol
0,methane,-5.551115e-17
1,ethane,-2.220446e-16


## Meaning and limits


The bubble pressure and flash split are source-parameter predictions with
unquantified model uncertainty. Agreement with the retained Engine baseline
is numerical verification. No measured mixture data or global phase-stability
certificate is used in this example.
